# Serverless job lifecycle

Runs one GPU job on Nebius Serverless AI Jobs and walks through its lifecycle: build the recipe, dry-run it, create it, check its status, wait for it, read its logs, reload it by id, and cancel it.

`examples/notebook/job_quickstart.ipynb` is the shorter version: create and wait only.

**Before you start**

- `uv sync` in the repo root, then start Jupyter from the same environment, for example `uv run --with jupyter jupyter lab`.
- A Nebius CLI profile (`nebius profile create`). The active profile supplies the project. The region comes from the profile or the project.
- Quota for one on-demand H100 in `eu-north1`. Change `gpu=` for another region; the alias table is in `docs/reference.md`.

Jupyter already runs an event loop, so every call that talks to Nebius uses the awaited twin (`create_async`, `wait_async`, …). A synchronous `create()` in this kernel raises `ValidationError` and names the twin to use.

In [2]:
import nebius_serverless as ns

ns.diagnostics()

{'version': '0.0.1',
 'python': '3.12.12',
 'platform': 'macOS-27.0.1-arm64-arm-64bit',
 'presentation': 'plain'}

## 1. Describe the job

A `Job` is a local recipe. Constructing it does not contact Nebius.

The command is a list of strings. To run a shell pipeline, pass it to `bash -lc`. `env` values are sent on create and never shown in dry-run output or error text.

In [ ]:
job = ns.Job(
    [
        "bash",
        "-lc",
        "nvidia-smi --query-gpu=name,memory.total --format=csv"
        ' && echo "hello from $GREETING"',
    ],
    image="nvidia/cuda:13.1.1-runtime-ubuntu24.04",
    gpu="h100",
    timeout="1h",
    env={"GREETING": "a notebook"},
    name="notebook-lifecycle",
)

{
    "name": job.name,
    "command": list(job.command),
    "image": job.image,
    "gpu": job.gpu,
    "pricing": job.pricing,
    "timeout_seconds": job.timeout_seconds,
}

{'name': 'notebook-lifecycle',
 'command': ['bash',
  '-lc',
  'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"'],
 'image': 'nvidia/cuda:13.1.1-runtime-ubuntu24.04',
 'gpu': 'h100',
 'pricing': 'on-demand',
 'timeout_seconds': 3600}

## 2. Dry run

Resolves the project, region, platform, and preset, and returns the create request. No job is created. Check the platform and preset before spending GPU time.

In [6]:
await job.create_async(dry_run=True)

{'image': 'nvidia/cuda:13.1.1-runtime-ubuntu24.04',
 'command': ['bash',
  '-lc',
  'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"'],
 'args': '-lc \'nvidia-smi --query-gpu=name,memory.total --format=csv && echo "hello from $GREETING"\'',
 'project': 'project-e00chatepr006ppncgsanv',
 'region': 'eu-north1',
 'platform': 'gpu-h100-sxm',
 'preset': '1gpu-16vcpu-200gb',
 'timeout': 3600,
 'pricing': 'on-demand',
 'env': ['GREETING'],
 'name': 'notebook-lifecycle',
 'disk': '250Gi',
 'shm': '16Gi'}

## 3. Create

Returns a `Run` as soon as Nebius accepts the job and its id is known. It does not wait for the container. **This cell starts a billable job.**

Save `run.id`. It is the only handle you need to find the job again from another notebook or terminal.

In [ ]:
run = await job.create_async()

job_id = run.id
job_id, run.status

## 4. Check status without blocking

`get_async` fetches the job once and stores the result on `run.record` and `run.status`. Re-run this cell to watch the state change. A failed container is data here, not an exception.

In [ ]:
await run.get_async()
run.status

## 5. Wait for a terminal state

Polls about every 10 seconds until the state is `COMPLETED`, `FAILED`, `CANCELLED`, or `ERROR`.

`timeout=` here limits only the local wait. When it expires, `WaitTimeout` is raised and the job keeps running in the cloud. Re-run the cell to keep waiting.

In [ ]:
try:
    await run.wait_async(timeout=30 * 60)
except ns.WaitTimeout as exc:
    print(f"Stopped waiting; job {exc.run_id} is still {run.status}.")
    print("Re-run this cell to keep waiting, or cancel it in step 8.")
else:
    run.display()

## 6. Read the logs

This client does not stream container logs. The Nebius CLI does. IPython substitutes `{job_id}` into the shell command. You should see the GPU name and `hello from a notebook`.

In [ ]:
!nebius ai job logs {job_id}

## 7. Reload by id

`jobs.get_async` starts from an id string: after a kernel restart, in another notebook, or for a job created from the CLI. It fetches once and never creates a job. `record` is the raw job JSON.

In [ ]:
loaded = await ns.jobs.get_async(job_id)

print(loaded.id, loaded.name, loaded.status)
sorted(loaded.record)

## 8. Cancel

Interrupting a cell, a local `WaitTimeout`, and restarting the kernel all leave the job running. Only `cancel_async` stops it. It returns once the request is accepted; a later `get_async` shows `CANCELLED`.

The cell skips the request when the job has already finished.

In [ ]:
TERMINAL = {"COMPLETED", "FAILED", "CANCELLED", "ERROR"}

await run.get_async(quiet=True)

if run.status in TERMINAL:
    print(f"Job {run.id} already finished: {run.status}")
else:
    await run.cancel_async()
    await run.get_async()
    print(run.status)